## FTP to Drupal Exports

This file contains the updated script for prepping a folder-level export between FTP and Drupal.

Note that this script continues to use the FTP API, but it is not really necessary anymore. The API is largely important to extract the unique FTP work ID from each work - since we are doing folder-level exports and each folder is now one work, it is just as easy if not easier to just look up the work ID and manually label each export.

In [44]:
# import required packages

import os
import dotenv

import requests
import pandas as pd
import json
import re
import numpy as np

from bs4 import BeautifulSoup
from collections import defaultdict

from google import genai
from google.genai import types
from concurrent.futures import ThreadPoolExecutor

### Collecting FTP transcriptions

Using "Option 2: the new way" from `ftp-api-folderlevel.ipynb`. Refer to that script for "the old way" if you prefer it.

In [45]:
# instead of changing the collection slug and querying the API etc etc, go to FTP and look up the work ID.
# For a detailed explanation of where to find it, see documentation

# then, uncomment the below code and fill in the info

label = 'Box 18 Folder 4' # replace with box/folder number of desired export
cut = '32237589' # replace with unique FTP work ID of desired export (important that this is a string)

In [46]:
new_url = f'https://fromthepage.com/iiif/{cut}/export/html' # url that hosts the html export
final = requests.get(new_url) # get request on html export url
html = final.text

In [47]:
def extract_pages(string):
    soup = BeautifulSoup(string, "lxml")

    # Find all <div> tags where id starts with "page-"
    page_divs = soup.find_all("div", id=re.compile(r"^page-\d+"))

    pages = []

    for page_div in page_divs:
        # Extract title from the <a name="..."> tag inside the page
        title_tag = page_div.find("a", attrs={"name": True})
        title = title_tag.get_text(strip=True) if title_tag else None

        # Extract page content
        content_tag = page_div.find("div", class_="page-content")
        content = content_tag.decode_contents() if content_tag else None

        # Extract all usernames from <small class="page-version-username">
        user_tags = page_div.find_all("small", class_="page-version-username")
        users = [tag.get_text(strip=True) for tag in user_tags]

        pages.append({
            "FTP_page_id": page_div.get("id"),
            "title": title,
            "content": content,
            "users": users
        })

    return pages

In [48]:
pages = extract_pages(html)

In [49]:
for page in pages:
    page['title'] = page['title'].strip(' cont.').strip(',')
    page['PJB'] = page['title'].split(',')[-1].strip(' ')
    # print(page['PJB'])

In [50]:
grouped = defaultdict(list)
for page in pages:
    grouped[page['PJB']].append(page)

merged_pages = [
    {
        'PJB': PJB,
        'content': ' '.join(p['content'] for p in group if p['content']),
        'users': sorted(set(u for p in group for u in p['users'])),
        'FTP_page_ids': [p['FTP_page_id'] for p in group]
    }
    for PJB, group in grouped.items()
]

In [51]:
for page in merged_pages:

    # get rid of newline characters
    page['content'] = page['content'].replace('\n','')

    # add contributor language in the appropriate place
    contributors = page.pop('users')
    contributors = ', '.join(contributors)
    page['content'] = page['content'] + "<p>Thanks to FromThePage transcription contributors: " + contributors + "</p>"

**For longer documents:** you may need to write them to individual text files and paste them in manually. There is sometimes a character limit in what CSVs can display/import. 2700 is an example that worked well here, but when running this script normally you should raise it (maybe 10000? IDK what a good number is)

In [52]:
to_remove = []
for page in merged_pages:
    if len(page['content']) >= 2700: # this obviously can and should be higher, but just as an example
        pjb_id = page['PJB']
        text = page['content']
        with open(f"{pjb_id}_output.txt", "w", encoding="utf-8") as f:
            f.write(text)
        to_remove.append(page)
    
for page in to_remove:
    merged_pages.remove(page)

In [53]:
export = pd.DataFrame(merged_pages)

In [54]:
# take a look at the export to make sure everything looks alright
export.head()

,PJB,content,FTP_page_ids
0,PJB 9658,"<p>July 2, 1974</p><p>Dear Richard:</p><p>Than...",[page-35359377]
1,PJB 9659,"<p>July 2, 1974</p><p>Dear Josie:</p><p>Just a...",[page-35359378]
2,PJB 9660,"<p>July 2, 1974</p><p>Dear Mr. Forbes:</p><p>T...",[page-35359379]
3,PJB 9661,"<p>July 12, 1974</p><p>Morris-Fallaize Insuran...",[page-35359380]
4,PJB 9662,"<p>July 9, 1974</p><p>Dear Bob,</p><p>In refer...",[page-35359381]


### Segmenting exports with Gemini

This is the new new stuff: eliminating the need for manual segmentation during single verification.

This runs with apparent success on Gemini 2.5-flash. Thinking of upgrading to Gemini 3.1-flash.

*Note: this script currently does not segment any longer documents that may have been removed due to the character limit in the above cells.*

In [55]:
# turn "content" column from export into a list
my_strings = export.content.to_list()

In [56]:
# FOR USERS USING DOTENV FILE:

# change to the directory where the dotenv file is (unique for each person)
os.chdir("/Users/charl/JBPP")

# load in stuff hidden in the .env file
dotenv.load_dotenv()
API_key = os.getenv('GEMINI_key')

# ELSE: comment out the three lines above

# API_key = 'YOUR_API_KEY_HERE'

In [57]:
# initialize client

client = genai.Client(api_key=API_key)

In [59]:
# Batched version here... seems to make sense to me. Could still possibly encounter token limits.

# define instructions and JSON schema for return. Be specific but not overwordy.

parsing_instructions = """
I am going to provide you a series of transcribed letters, represented as strings with HTML tags.
Break down each of the following strings into the following component parts, including the appropriate HTML.
Do not leave any of the content of the original string out, and do not add anything new. Do not place any content in multiple fields.
If a component part does not have a corresponding part of the original string, return a null value.

- document_top (string). Anything that appears above the dateline.
- dateline (string)
- salutation (string)
- document_body (string). Include the closing valediction in this component.
- signature (string)
- inside_address (string)
- transcriber_attribution (string)
- document_extra (string). Anything that does not fall into the above categories.
"""

# 1. Configuration

BATCH_SIZE = 10  # How many strings per prompt
MAX_WORKERS = 5  # How many simultaneous API calls

def process_batch(batch_strings):
    """Function to process a single chunk of strings"""
    prompt = f"{parsing_instructions}\n\nStrings to process:\n{batch_strings}"
    
    try:
        response = client.models.generate_content(
            model='gemini-3-flash-preview',
            contents=prompt,
            config={'response_mime_type': 'application/json'}
        )
        
        # Load the JSON from this specific response
        return json.loads(response.text)
    except Exception as e:
        print(f"Error processing batch: {e}")
        return []

# 2. Split 'my_strings' into smaller chunks

chunks = [my_strings[i:i + BATCH_SIZE] for i in range(0, len(my_strings), BATCH_SIZE)]

# 3. Use ThreadPoolExecutor to run batches in parallel

all_results = []

print(f"Starting processing {len(chunks)} batches...")

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # This maps the function to all chunks across multiple threads
    results = list(executor.map(process_batch, chunks))

# 4. Flatten the results and create DataFrame
# Results is a list of lists (each batch returns a list of objects)

flattened_results = [item for sublist in results for item in sublist]

parsed_df = pd.DataFrame(flattened_results)

print(f"Processed {len(parsed_df)} total rows.")

Starting processing 1 batches...
Done! Processed 10 total rows.


#### Batched Token tracking

In progress

In [ ]:
# 1. Configuration
BATCH_SIZE = 10
MAX_WORKERS = 5

# Initialize a dictionary to track global usage
usage_stats = {
    "prompt_tokens": 0,
    "candidates_tokens": 0,
    "total_tokens": 0
}

def process_batch(batch_strings):
    prompt = f"{parsing_instructions}\n\nStrings to process:\n{batch_strings}"
    
    try:
        response = client.models.generate_content(
            model='gemini-2.0-flash',
            contents=prompt,
            config={'response_mime_type': 'application/json'}
        )
        
        # Extract Token Metadata
        usage = response.usage_metadata
        batch_tokens = {
            "prompt": usage.prompt_token_count,
            "candidates": usage.candidates_token_count,
            "cached": usage.cached_content_token_count,
            "thinking": usage.thoughts_token_count
        }
        
        return {
            "data": json.loads(response.text),
            "tokens": batch_tokens,
            "success": True
        }
    except Exception as e:
        print(f"Error processing batch: {e}")
        return {"data": [], "tokens": None, "success": False}

# 2. Prepare chunks
chunks = [my_strings[i:i + BATCH_SIZE] for i in range(0, len(my_strings), BATCH_SIZE)]

# 3. Process
all_results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    # Run batches
    batch_responses = list(executor.map(process_batch, chunks))

# 4. Aggregate Data and Tokens
for resp in batch_responses:
    if resp["success"]:
        # Add data to master list
        all_results.extend(resp["data"])
        
        # Add tokens to global stats
        usage_stats["prompt_tokens"] += resp["tokens"]["prompt"]
        usage_stats["candidates_tokens"] += resp["tokens"]["candidates"]
        usage_stats["total_tokens"] += resp["tokens"]["total"]

# 5. Output Results
parsed_df = pd.DataFrame(all_results)

print("-" * 30)
print(f"PROCESSING COMPLETE")
print(f"Total Rows Parsed: {len(parsed_df)}")
print(f"Input Tokens:      {usage_stats['prompt_tokens']:,}")
print(f"Output Tokens:     {usage_stats['candidates_tokens']:,}")
print(f"Total Tokens:      {usage_stats['total_tokens']:,}")
print("-" * 30)

#### Quick look at token usage

Don't need to run this, but is handy to keep an eye on expenses

In [60]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  cache_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=977
    ),
  ],
  cached_content_token_count=977,
  candidates_token_count=2221,
  prompt_token_count=1843,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=1843
    ),
  ],
  thoughts_token_count=17978,
  total_token_count=22042
)

In [36]:
# guess on tokens here:
# input (cached + prompt), output (candidates + thoughts)

977 + 1843, 2221+17978

(2820, 20199)

In [43]:
# rough price here: input, output
# 0.3 USD per million input and 2.5 USD per million is current pricing for Gemini 2.5-flash as of 04/14/2026

2820/1000000 * 0.3, 20199/1000000 * 2.5

(0.000846, 0.0504975)

In [39]:
# from a perspective of employee time versus cost... this would save a lot a lot.
# I bet I could save some thinking tokens if I spelled things correctly lol
# do not think more specific directions are needed and would inevitably require spending more thinking tokens

#### Sorting things

In [61]:
parsed_df.head()

,document_top,dateline,salutation,document_body,signature,inside_address,transcriber_attribution,document_extra
0,None,"<p>July 2, 1974</p>",<p>Dear Richard:</p>,<p>Thanks for the letter. I am glad that you ...,<p>Julian Bond</p>,"<p>Mr. Richard King, Director<br/>United Negro...",<p>Thanks to FromThePage transcription contrib...,<p>j</p>
1,None,"<p>July 2, 1974</p>",<p>Dear Josie:</p>,<p>Just a quick note to say we have arranged f...,<p>Julian Bond</p>,<p>Ms. Josie Davis<br/>Holston Distributor<br/...,<p>Thanks to FromThePage transcription contrib...,<p>j</p>
2,None,"<p>July 2, 1974</p>",<p>Dear Mr. Forbes:</p>,<p>Thanks for writing back. I admire your spi...,<p>Julian Bond</p>,<p>Mr. Ray T. Forbes<br/>789 Ponce De Leon Ave...,<p>Thanks to FromThePage transcription contrib...,<p>J</p>
3,None,"<p>July 12, 1974</p>",<p>Gentlemen:</p>,<p>Under the terms of the policy numbered abov...,<p>Julian Bond</p>,<p>Morris-Fallaize Insurance Inc.<br/>1252 W. ...,<p>Thanks to FromThePage transcription contrib...,<p>re: policy LPF 002056</p><p>enc 2</p>
4,None,"<p>July 9, 1974</p>","<p>Dear Bob,</p>","<p>In reference to paragraph 1, detailing the ...",<p>Julian Bond</p>,<p>Robert Walker<br/>American Program Bureau<b...,<p>Thanks to FromThePage transcription contrib...,None


In [62]:
final_df = (
    export[['PJB', 'FTP_page_ids']]
    .join(parsed_df[['document_top', 'salutation', 'document_body', 'signature', 'transcriber_attribution', 'document_extra']])
    .assign(
        dateline = parsed_df['dateline'] + parsed_df['inside_address']
    )
)

In [63]:
final_df.head()

,PJB,FTP_page_ids,document_top,salutation,document_body,signature,transcriber_attribution,document_extra,dateline
0,PJB 9658,[page-35359377],None,<p>Dear Richard:</p>,<p>Thanks for the letter. I am glad that you ...,<p>Julian Bond</p>,<p>Thanks to FromThePage transcription contrib...,<p>j</p>,"<p>July 2, 1974</p><p>Mr. Richard King, Direct..."
1,PJB 9659,[page-35359378],None,<p>Dear Josie:</p>,<p>Just a quick note to say we have arranged f...,<p>Julian Bond</p>,<p>Thanks to FromThePage transcription contrib...,<p>j</p>,"<p>July 2, 1974</p><p>Ms. Josie Davis<br/>Hols..."
2,PJB 9660,[page-35359379],None,<p>Dear Mr. Forbes:</p>,<p>Thanks for writing back. I admire your spi...,<p>Julian Bond</p>,<p>Thanks to FromThePage transcription contrib...,<p>J</p>,"<p>July 2, 1974</p><p>Mr. Ray T. Forbes<br/>78..."
3,PJB 9661,[page-35359380],None,<p>Gentlemen:</p>,<p>Under the terms of the policy numbered abov...,<p>Julian Bond</p>,<p>Thanks to FromThePage transcription contrib...,<p>re: policy LPF 002056</p><p>enc 2</p>,"<p>July 12, 1974</p><p>Morris-Fallaize Insuran..."
4,PJB 9662,[page-35359381],None,"<p>Dear Bob,</p>","<p>In reference to paragraph 1, detailing the ...",<p>Julian Bond</p>,<p>Thanks to FromThePage transcription contrib...,None,"<p>July 9, 1974</p><p>Robert Walker<br/>Americ..."


In [64]:
final_df.to_csv(f'export_{label}.csv', index=False)

PermissionError: [Errno 13] Permission denied: 'export_Box 18 Folder 4.csv'